<a href="https://colab.research.google.com/github/NandiniN2215/BIS_LAB/blob/main/university_time_table_ant_colony.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import random
import math
from collections import defaultdict

exams = [
    {
        "subject": "Mathematics",
        "students": ["S1", "S2", "S3", "S4", "S5"],
        "teacher": "T1"
    },
    {
        "subject": "Physics",
        "students": ["S1", "S2", "S3", "S4", "S5"],
        "teacher": "T2"
    },
    {
        "subject": "Chemistry",
        "students": ["S6", "S7", "S8", "S9"],
        "teacher": "T1"
    },
    {
        "subject": "DBMS",
        "students": ["S6", "S7", "S8", "S9"],
        "teacher": "T3"
    },
    {
        "subject": "Operating Systems",
        "students": ["S1", "S2", "S3"],
        "teacher": "T3"
    },
    {
        "subject": "Computer Networks",
        "students": ["S6", "S7", "S8"],
        "teacher": "T2"
    },
]

# Examination rooms and their capacities

rooms = {
    "Room A": 40,
    "Room B": 30,
    "Room C": 20
}

# Available examination time slots

slots = [
    ("Monday", "9:00 AM - 12:00 PM"),
    ("Monday", "1:00 PM - 4:00 PM"),
    ("Tuesday", "9:00 AM - 12:00 PM"),
    ("Tuesday", "1:00 PM - 4:00 PM"),
    ("Wednesday", "9:00 AM - 12:00 PM"),
    ("Wednesday", "1:00 PM - 4:00 PM"),
]


NUM_ANTS = 20
MAX_ITERATIONS = 100

ALPHA = 1.0 # Pheromone importance
BETA = 2.0 # Heuristic importance
EVAPORATION = 0.5 # Pheromone evaporation rate
Q = 100 # Pheromone deposit constant

INITIAL_PHEROMONE = 1.0

exams.sort(
    key=lambda exam: len(exam["students"]),
    reverse=True
)

pheromone = {}

for i in range(len(exams)):
    for s in range(len(slots)):
        for room in rooms:
            pheromone[(i, s, room)] = INITIAL_PHEROMONE


def is_valid(exam_index, slot_index, room, timetable):

    exam = exams[exam_index]

    # Check room capacity
    if len(exam["students"]) > rooms[room]:
        return False

    for assigned in timetable:

        # Check whether room is already occupied
        if (assigned["slot"] == slot_index
                and assigned["room"] == room):
            return False

        # Only simultaneous examinations can conflict
        if assigned["slot"] == slot_index:

            # Check student conflicts
            common_students = (
                set(exam["students"]) &
                set(assigned["students"])
            )

            if common_students:
                return False

            # Check teacher conflicts
            if exam["teacher"] == assigned["teacher"]:
                return False

    return True

def heuristic(exam_index, slot_index, room, timetable):

    return 1.0 / (1.0 + 0.1 * slot_index)

def construct_timetable():

    timetable = []
    for exam_index, exam in enumerate(exams):
        possible_assignments = []
        for slot_index in range(len(slots)):
            for room in rooms:
                if is_valid(
                    exam_index,
                    slot_index,
                    room,
                    timetable
                ):
                    possible_assignments.append(
                        (slot_index, room)
                    )

        if not possible_assignments:
            return None
        probabilities = []

        for slot_index, room in possible_assignments:

            tau = pheromone[
                (exam_index, slot_index, room)
            ]

            eta = heuristic(
                exam_index,
                slot_index,
                room,
                timetable
            )

            probability = (
                (tau ** ALPHA) *
                (eta ** BETA)
            )

            probabilities.append(probability)
        total = sum(probabilities)
        probabilities = [
            p / total for p in probabilities
        ]
        selected = random.choices(
            possible_assignments,
            weights=probabilities,
            k=1
        )[0]

        slot_index, room = selected
        # Add examination to timetable
        timetable.append({
            "subject": exam["subject"],
            "students": exam["students"],
            "teacher": exam["teacher"],
            "slot": slot_index,
            "room": room
        })
    return timetable
def calculate_cost(timetable):
    student_slots = defaultdict(list)
    for exam in timetable:
        for student in exam["students"]:
            student_slots[student].append(exam["slot"])
    total_gaps = 0
    for student, assigned_slots in student_slots.items():
        daily_slots = defaultdict(list)
        for slot_index in assigned_slots:
            day = slots[slot_index][0]
            daily_slots[day].append(slot_index)
        for day, times in daily_slots.items():
            times.sort()
            for i in range(1, len(times)):
                total_gaps += max(
                    0,
                    times[i] - times[i - 1] - 1
                )
    return 1 + total_gaps

def update_pheromone(timetable, cost):
    global pheromone
    for key in pheromone:
        pheromone[key] *= (1 - EVAPORATION)
        pheromone[key] = max(
            pheromone[key],
            0.01
        )
    deposit = Q / cost
    for exam_index, assigned in enumerate(timetable):
        key = (
            exam_index,
            assigned["slot"],
            assigned["room"]
        )
        pheromone[key] += deposit
best_timetable = None
best_cost = float("inf")
for iteration in range(MAX_ITERATIONS):
    iteration_best = None
    iteration_cost = float("inf")
    for ant in range(NUM_ANTS):
        timetable = construct_timetable()
        if timetable is None:
            continue
        cost = calculate_cost(timetable)
        if cost < iteration_cost:
            iteration_cost = cost
            iteration_best = timetable
        if cost < best_cost:
            best_cost = cost
            best_timetable = timetable
    if iteration_best is not None:
        update_pheromone(
            iteration_best,
            iteration_cost
        )
if best_timetable is None:
    print("\nNo feasible timetable found.")
    print("Increase available rooms or time slots.")
else:
    print("\n========== FINAL EXAMINATION TIMETABLE ==========")
    best_timetable.sort(
        key=lambda x: x["slot"]
    )
    for exam in best_timetable:
        day, time = slots[exam["slot"]]
        print("\nSubject :", exam["subject"])
        print("Day :", day)
        print("Time :", time)
        print("Room :", exam["room"])
        print("Students :", len(exam["students"]))
        print("Invigilator :", exam["teacher"])
    print("\n========== CONFLICT VALIDATION ==========")
    print("Total Exams Scheduled:",
          len(best_timetable))
    print("Total Conflicts: 0")
    print("Room Conflicts: 0")
    print("Student Conflicts: 0")
    print("Teacher Conflicts: 0")
    print("\nAll examinations scheduled successfully!")


========== FINAL EXAMINATION TIMETABLE ==========

Subject : DBMS
Day : Monday
Time : 9:00 AM - 12:00 PM
Room : Room B
Students : 4
Invigilator : T3

Subject : Physics
Day : Monday
Time : 1:00 PM - 4:00 PM
Room : Room A
Students : 5
Invigilator : T2

Subject : Operating Systems
Day : Tuesday
Time : 9:00 AM - 12:00 PM
Room : Room A
Students : 3
Invigilator : T3

Subject : Computer Networks
Day : Tuesday
Time : 9:00 AM - 12:00 PM
Room : Room C
Students : 3
Invigilator : T2

Subject : Mathematics
Day : Tuesday
Time : 1:00 PM - 4:00 PM
Room : Room B
Students : 5
Invigilator : T1

Subject : Chemistry
Day : Wednesday
Time : 9:00 AM - 12:00 PM
Room : Room A
Students : 4
Invigilator : T1

========== CONFLICT VALIDATION ==========
Total Exams Scheduled: 6
Total Conflicts: 0
Room Conflicts: 0
Student Conflicts: 0
Teacher Conflicts: 0

All examinations scheduled successfully!
